In [5]:
import subprocess, sys

VLLM_PIN = "0.28.0"
TRANSFORMERS_PIN = "5.16.1"
ACCELERATE_PIN = "1.14.0"
HTTPX_PIN = "0.28.1"
OPENAI_PIN = "3.7.0"

def pip_install(specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [6]:
pip_install([
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
])
print("serving pins installed (updated for Python 3.13)")

installing: vllm==0.28.0 transformers==5.16.1 accelerate==1.14.0 httpx==0.28.1 openai==3.7.0
serving pins installed (updated for Python 3.13)


In [7]:
import subprocess
subprocess.run([
    "pip", "install", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu130",
    "--force-reinstall"
], check=True)

CompletedProcess(args=['pip', 'install', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu130', '--force-reinstall'], returncode=0)

In [11]:
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 5594, logging to /content/server.log


In [12]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 105s: http://localhost:8000/v1/models -> 200


In [13]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one sentence, what is a GPU?"}],
)
print(r.choices[0].message.content)

A GPU (Graphics Processing Unit) is a specialized chip designed to accelerate the performance of computer graphics and scientific computing tasks.


In [14]:
from google.colab import files
uploaded = files.upload()   # pick baselines.json
import json
baseline = json.load(open("baselines.json"))
print("baseline batch tokens/s:", baseline["batch"])

Saving baselines.json to baselines.json
baseline batch tokens/s: {'1': 35.9, '4': 52.1, '8': 103.3}


In [21]:
# Async A/B client for Lab W3D3 (engine swap).
# Paste the whole file as one Colab cell (after the vLLM server is healthy), then
# call run_sweep(...) as the day-3 README shows. It fires N concurrent chat
# completions per level with httpx + asyncio, excludes a warm-up round, and
# reports aggregate tokens/s at each concurrency level.
#
# It talks to the OpenAI-compatible /v1 endpoint, so the same client works
# against any team's service. No secrets: the local vLLM server needs no key.

import asyncio
import time

import httpx

# A fixed prompt set so every run measures the same work. Varied lengths, no
# duplicates. Requests cycle through this list.
FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

# Output lengths per request, cycled in order. This list is IDENTICAL to Monday's
# QUEUE in the day-2 lab, and it has to stay that way: the A/B is only honest if
# both engines are asked for exactly the same work. 24 requests, 18 that want 32
# tokens and 6 that want 256, so a long request is always in flight alongside
# short ones.
#
# The mixed lengths are the entire point. Ask every request for the same number
# of tokens and there is no straggler, static batching pays no tax, and
# continuous batching has nothing to win back. You would measure a flat speedup
# across concurrency and conclude, wrongly, that continuous batching does not
# scale.
QUEUE = [32, 32, 32, 256] * 6

# Fallback when a caller does not pass a length.
MAX_TOKENS = 128
# Warm-up requests per level, dropped from the timing.
WARMUP = 4


async def _one_request(client, base_url, model, prompt, max_tokens=MAX_TOKENS):
    """Fire one chat completion, return the count of completion tokens."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    r = await client.post(f"{base_url}/chat/completions", json=payload)
    r.raise_for_status()
    body = r.json()
    usage = body.get("usage", {})
    # completion_tokens is what the server generated; fall back to counting.
    # Accounting note vs Monday: static_queue counted REQUESTED tokens, which
    # equals generated there (greedy decode runs to the cap). vLLM can stop at
    # EOS short of the cap, so counting usage is the honest number for it -
    # any bias this introduces runs AGAINST vLLM, never for it.
    ct = usage.get("completion_tokens")
    if ct is None:
        ct = len(body["choices"][0]["message"]["content"].split())
    return ct


async def _run_level(client, base_url, model, prompts, concurrency, total_requests):
    """Run total_requests requests, at most `concurrency` in flight at once."""
    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(client, base_url, model, prompt, max_tokens)

    # Each request carries its own output length from QUEUE, so the workload
    # matches Monday's static-batching baseline request for request.
    tasks = [asyncio.create_task(guarded(prompts[i % len(prompts)],
                                         QUEUE[i % len(QUEUE)]))
             for i in range(total_requests)]
    t0 = time.time()
    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)
    dt = time.time() - t0
    total_tokens = sum(counts)
    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }


async def run_sweep(base_url, model, prompts=FIXED_PROMPTS,
                    concurrencies=(1, 4, 8), requests_per_level=24):
    """Sweep the concurrency levels; return a list of per-level result dicts.

    A warm-up round runs first and is discarded so model-load and cache-warm
    cost stays out of the measured numbers.
    """
    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        # warm-up: fire WARMUP requests, ignore timing
        await asyncio.gather(*[
            _one_request(client, base_url, model, prompts[i % len(prompts)])
            for i in range(WARMUP)
        ])
        for c in concurrencies:
            level = await _run_level(client, base_url, model, prompts, c,
                                     requests_per_level)
            print("level:", level)
            results.append(level)
    return results


In [22]:
prompts = FIXED_PROMPTS
vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=prompts,
    concurrencies=[1, 4, 8],
)
for level in vllm_measured:
    print(level)

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 62.9, 'wall_s': 22.739}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 177.7, 'wall_s': 8.055}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 268.0, 'wall_s': 5.339}
{'concurrency': 1, 'requests': 24, 'tokens_per_s': 62.9, 'wall_s': 22.739}
{'concurrency': 4, 'requests': 24, 'tokens_per_s': 177.7, 'wall_s': 8.055}
{'concurrency': 8, 'requests': 24, 'tokens_per_s': 268.0, 'wall_s': 5.339}


In [23]:
import json

def tokps_at(level_list, c):
    return next(x["tokens_per_s"] for x in level_list if x["concurrency"] == c)

vllm_by_c = {x["concurrency"]: x["tokens_per_s"] for x in vllm_measured}
# Monday's static batching at 1/4/8 is the baseline curve
base_by_c = {int(k): v for k, v in baseline["batch"].items()}

speedup = {c: round(vllm_by_c[c] / base_by_c[c], 2)
           for c in vllm_by_c if c in base_by_c}

report = {
    "baseline": base_by_c,               # from Monday's baselines.json
    "vllm": vllm_by_c,                    # measured today
    "speedup_by_concurrency": speedup,
    "predicted_speedup": None,           # <- put your prediction-card number here
}
with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "baseline": {
    "1": 35.9,
    "4": 52.1,
    "8": 103.3
  },
  "vllm": {
    "1": 62.9,
    "4": 177.7,
    "8": 268.0
  },
  "speedup_by_concurrency": {
    "1": 1.75,
    "4": 3.41,
    "8": 2.59
  },
  "predicted_speedup": null
}


In [24]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]
print(f"static batching scales {static_scaling:.2f}x, vLLM scales {vllm_scaling:.2f}x")
print(f"continuous batching is worth {vllm_scaling / static_scaling:.2f}x of scaling")

static batching scales 2.88x, vLLM scales 4.26x
continuous batching is worth 1.48x of scaling


In [25]:
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    # give it a moment, then confirm the port is free
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

In [26]:
# Green-check verifier for Lab W3D3 (engine swap).
# Paste this as the last cell of your day-3 notebook and run it. It reads
# ab_report.json and checks the schema, that vLLM's concurrency-8 throughput
# beats Monday's batch-8 baseline, and that the speedup fields were computed.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"
    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")
    try:
        with open(path) as fh:
            report = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail("baseline must be a non-empty object (from Monday's baselines.json)")
    if not isinstance(vllm, dict) or not vllm:
        fail("vllm must be a non-empty object of measured throughput")
    if not isinstance(speedup, dict) or not speedup:
        fail("speedup_by_concurrency must be a non-empty object")

    # keys may be strings or ints depending on how the report was built; normalise
    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)
    if base8 is None:
        fail("baseline has no concurrency-8 (batch-8) number")
    if vllm8 is None:
        fail("vllm has no concurrency-8 number")
    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail("concurrency-8 throughput values must be numbers")

    # the headline claim of the day
    if not vllm8 > base8:
        fail(f"vllm concurrency-8 throughput ({vllm8}) not above baseline "
             f"batch-8 ({base8}); the engine swap should win here")

    # speedup fields must be computed (present and numeric for at least c=8)
    s8 = get_c(speedup, 8)
    if s8 is None or not isinstance(s8, (int, float)):
        fail("speedup_by_concurrency has no numeric value at concurrency 8")
    # sanity: the reported speedup should match vllm8/base8 within rounding
    expected = vllm8 / base8
    if abs(s8 - expected) > 0.1:
        fail(f"speedup at 8 ({s8}) does not match vllm/baseline "
             f"({expected:.2f}); recompute it")

    print(f"baseline batch-8: {base8}, vllm concurrency-8: {vllm8}")
    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)

baseline batch-8: 103.3, vllm concurrency-8: 268.0
speedup at 8: 2.59x
GREEN CHECK: PASS
